# 다층 퍼셉트론
다층 퍼셉트론(Multi-Layer Perceptron, MLP)은 단층 퍼셉트론을 여러 개 쌓아은닉층을 생성한다.
즉, 다층 퍼셉트론은 은닉층이 한 개 이상인 퍼셉트론 구조를 의미한다. 은닉층을 2개 이상 연결한다면
심층 신경망(Deep Neural Network, DNN)이라 부른다. 

결국 다층 퍼셉트론은 역전파 과
정을 통해 모든 노드의 가중치와 편향을 수정해 오차가 작아지는 방향으로 학습이 진행된다. 학습 방법
을 다시 정리하면 다음과 같다.

1. 입력층부터 출력층까지 순전파를 진행
2. 출력값(예측값)과실젯값으로오차계산
3. 오차를 퍼셉트론의 역방향으로 보내면서 입력된 노드의 기여도 측정
A. 손실 함수를 편미분해 기울기 계산
B. 연쇄 법칙(Chain R니Ie)을 통해 기울기를 계산
4. 입력층에 도달할 때까지 노드의 기여도 측정
5. 모든 가중치에 최적화 알고리즘 수행

![nn](image/solvedXor.png)

In [1]:
import torch
import pandas as pd
from torch import nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

In [2]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x1 = df.iloc[:, 0].values
        self.x2 = df.iloc[:, 1].values
        self.y = df.iloc[:, 2].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x1[index], self.x2[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y

    def __len__(self):
        return self.length

class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Linear(2, 2),
            nn.Sigmoid()
        )
        self.layer2 = nn.Sequential(
            nn.Linear(2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

In [3]:
train_dataset = CustomDataset("../datasets/perceptron.csv")
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

In [4]:

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CustomModel().to(device)
criterion = nn.BCELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [5]:
for epoch in range(10000):
    cost = 0.0

    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(train_dataloader)

    if (epoch + 1) % 1000 == 0:
        print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

C:\Users\H11\AppData\Local\Temp\ipykernel_24684\4096481949.py:10: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  x = torch.FloatTensor([self.x1[index], self.x2[index]])
C:\Users\H11\AppData\Local\Temp\ipykernel_24684\4096481949.py:11: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  y = torch.FloatTensor([self.y[index]])


Epoch : 1000, Cost : 0.690
Epoch : 2000, Cost : 0.642
Epoch : 3000, Cost : 0.324
Epoch : 4000, Cost : 0.068
Epoch : 5000, Cost : 0.034
Epoch : 6000, Cost : 0.023
Epoch : 7000, Cost : 0.017
Epoch : 8000, Cost : 0.013
Epoch : 9000, Cost : 0.011
Epoch : 10000, Cost : 0.009


단층 퍼셉트론 구조로 XOR 문제를 해결하려고 한다면, 비용이 더 이상 감소되지 않는 것을 확인할 수
있다. 이는 하나의 계층으로는 XOR 게이트 문제를 해결할 수 없어 발생하는 문제다.
모델에 값을 입력했을 때도 출력값이 0.5 내외로 출력돼 학습이 정상적으로 진행되지 않은 것을 확인할
수 있다. 그러므로 모델에 계층을 하나 더 추가해 다층 퍼셉트론 구조로 변경한다. 다음 예제 3.59는 모
델의 변경 사항을 보여준다.

예제 3.59에서는 XOR 문제를 해결하기 위해 계층을 하나 더 추가해 모델을 학습했다. 예제 3.58과 달
리, 학습이 진행될수록 비용이 감소하는 것을 확인할 수 있다. 또한, 모델에 값을 입력했을 때 출력값이
XOR 게이트의 출력값과 동일하다는 것도 확인할 수 있다.
퍼셉트론은 많은 머신러닝 애플리케이션에서 사용된다. 특히 이진 분류 작업에서 여전히 사용되는 간단
하고 효율적인 모델이다. 하지만 데이터의 복잡한 패턴을 학습할 수 없으며, 선형으로 분리되지 않는 데
이터를 분류할 수 없는 등 몇 가지 제한 사항이 있다. 이러한 제한으로 인해 보다 복잡한 작업에 더 적합
한 고급 신경망 모델이 개발됐다.

In [6]:
with torch.no_grad():
    model.eval()
    inputs = torch.FloatTensor([
        [0, 0],
        [0, 1],
        [1, 0],
        [1, 1]
    ]).to(device)
    outputs = model(inputs)
    
    print("---------")
    print(outputs)
    print(outputs <= 0.5)

---------
tensor([[0.0088],
        [0.9914],
        [0.9880],
        [0.0080]], device='cuda:0')
tensor([[ True],
        [False],
        [False],
        [ True]], device='cuda:0')
